# Choosing the index surgery

Before an index surgery can be defined we need to know whether "the surgery" is even unambiguous,
so this notebook counts the distinct calendar days carrying a surgical procedure for each patient
in the surgery x Fitbit cohort. Patients with one surgery-day need no index rule; the shape of
the tail says whether "first", "last" or "longest admission" is the consequential choice and how
many patients it moves.

**Governance.** Every query aggregates inside BigQuery and returns counts only, so no
participant-level rows enter the notebook. Unsuppressed figures stay git-ignored while
`*_suppressed.png` is committable, and the small-cell threshold must be confirmed against current
All of Us policy before anything is published.


# 1-setup


### Datasets, helpers and output directories

`srp_common` resolves the pinned CDR, resolves the cohort-builder index dataset, and creates this
notebook's git-ignored figure and cache directories. The repo root is found by walking up from the
working directory, so the notebook runs from either its task directory or the root.


In [ ]:
import sys
from pathlib import Path

import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common

cfg = srp_common.init("surgery_day_distribution")
CDR, CDR_INDEX = cfg.CDR, cfg.CDR_INDEX
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

### Cohort and surgical concept set

The surgical definition is concept `4301351` plus every descendant, and cohort membership is EHR
data and Fitbit data and at least one such procedure — read from the cohort-builder index tables
so this notebook cannot drift from the built cohort. The CTEs reduce qualifying rows to
person-days and then to a surgery-day count per patient, composed once into a prefix that every
query below reuses.


In [ ]:
SURGERY_ANCESTOR_CONCEPT_ID = 4301351

_SURGICAL_CONCEPT_CTE = f"""
    surgical_concept AS (
        SELECT descendant AS concept_id
        FROM `{CDR_INDEX}`.T_HAD_procedureConcept_default
        WHERE ancestor = {SURGERY_ANCESTOR_CONCEPT_ID}
        UNION ALL
        SELECT {SURGERY_ANCESTOR_CONCEPT_ID}
    )
"""

_COHORT_CTE = f"""
    cohort AS (
        SELECT id AS person_id
        FROM `{CDR_INDEX}`.T_ENT_person
        WHERE has_ehr_data = true
          AND has_fitbit = true
          AND id IN (
              SELECT person_id
              FROM `{CDR_INDEX}`.T_ENT_procedureOccurrence
              WHERE procedure_concept_id IN (SELECT concept_id FROM surgical_concept)
          )
    )
"""

_SURGICAL_EVENTS_CTE = f"""
    surgical_events AS (
        SELECT
            po.person_id,
            DATE(po.procedure_datetime) AS surgery_date
        FROM `{CDR}`.procedure_occurrence AS po
        WHERE po.person_id IN (SELECT person_id FROM cohort)
          AND po.procedure_concept_id IN (SELECT concept_id FROM surgical_concept)
          AND po.procedure_datetime IS NOT NULL
    )
"""

_SURGERY_DAYS_CTE = """
    surgery_days AS (
        SELECT person_id, COUNT(DISTINCT surgery_date) AS n_surgery_days
        FROM surgical_events
        GROUP BY person_id
    )
"""

_COHORT_CTES = "WITH " + ",".join([
    _SURGICAL_CONCEPT_CTE,
    _COHORT_CTE,
    _SURGICAL_EVENTS_CTE,
    _SURGERY_DAYS_CTE,
])

### Figure setup

One hue plus text and grid greys, since the distribution is a single series needing no legend, and
the hue is validated at or above 3:1 contrast on the light surface. `collapse_small_cells`
aggregates the sparse tail so no cell falls under the suppression threshold, and `figure_path`
derives the filename from the `suppress` flag so the `_suppressed` suffix cannot claim a
suppression that did not happen.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

SUPPRESS_THRESHOLD = 20

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_MUTED = "#52514e"
SERIES = "#2a78d6"
GRID = "#e3e2df"

_thousands = FuncFormatter(lambda v, _: f"{int(v):,}")

def collapse_small_cells(dist, threshold=SUPPRESS_THRESHOLD):
    """Return (kept_rows, tail_label, tail_patients, n_absorbed)."""
    d = dist.sort_values("n_surgery_days").reset_index(drop=True)
    keep = d.copy()
    tail_patients = 0
    absorbed = 0

    while len(keep) > 1 and (
        tail_patients and tail_patients < threshold
        or keep.iloc[-1]["n_patients"] < threshold
    ):
        tail_patients += int(keep.iloc[-1]["n_patients"])
        absorbed += 1
        keep = keep.iloc[:-1]

    if absorbed == 0:
        return keep, None, 0, 0

    first_absorbed = int(d.iloc[len(keep)]["n_surgery_days"])
    return keep, f"{first_absorbed}+", tail_patients, absorbed

def figure_path(name, suppress):
    return FIGURES / f"{name}{'_suppressed' if suppress else ''}.png"


def plot_surgery_days(dist, suppress=False, logy=False, save_as=None):
    d = dist.sort_values("n_surgery_days").reset_index(drop=True)
    tail_label, tail_patients, n_absorbed = None, 0, 0
    if suppress:
        d, tail_label, tail_patients, n_absorbed = collapse_small_cells(dist)
        d = d.copy()

    xs = list(range(len(d)))
    labels = [str(int(v)) for v in d["n_surgery_days"]]
    heights = [int(v) for v in d["n_patients"]]
    if tail_label:
        xs.append(len(d))
        labels.append(tail_label)
        heights.append(tail_patients)

    shown_total = sum(heights)
    cum_share = [sum(heights[: i + 1]) / shown_total for i in range(len(heights))]

    fig = plt.figure(figsize=(10, 8.2), facecolor=SURFACE)
    gs = fig.add_gridspec(3, 1, height_ratios=[0.5, 1.5, 1.0], hspace=0.55)

    ax_kpi = fig.add_subplot(gs[0])
    ax_kpi.set_facecolor(SURFACE)
    ax_kpi.axis("off")
    tiles = [
        (f"{total_patients:,}", "patients with a\ndated surgery-day"),
        (f"{median_days}", "median surgery-days\nper patient"),
        (f"{single_day / total_patients:.0%}", "have exactly\none surgery-day"),
        (f"{multi_day:,}", "have two or more\n(the ones a rule decides)"),
    ]
    for i, (value, label) in enumerate(tiles):
        x = i / len(tiles) + 0.01
        ax_kpi.text(x, 0.62, value, transform=ax_kpi.transAxes, fontsize=25,
                    fontweight="600", color=INK, va="center", ha="left")
        ax_kpi.text(x, 0.12, label, transform=ax_kpi.transAxes, fontsize=9.5,
                    color=INK_MUTED, va="center", ha="left", linespacing=1.5)

    ax = fig.add_subplot(gs[1])
    ax.set_facecolor(SURFACE)
    bars = ax.bar(xs, heights, width=0.78, color=SERIES, linewidth=0, zorder=3)
    ax.set_title("Patients by number of distinct surgery-days",
                 fontsize=13, fontweight="600", color=INK, loc="left", pad=10)
    ax.set_xlabel("distinct days with a surgical procedure", fontsize=10, color=INK_MUTED)
    ax.set_ylabel("patients", fontsize=10, color=INK_MUTED)
    if logy:
        ax.set_yscale("log")
    ax.yaxis.set_major_formatter(_thousands)
    ax.set_xticks(xs)
    ax.set_xticklabels(labels)
    for rect, height in zip(bars, heights):
        ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height(),
                f"{height:,}", ha="center", va="bottom",
                fontsize=8.5, color=INK_MUTED)

    ax2 = fig.add_subplot(gs[2])
    ax2.set_facecolor(SURFACE)
    ax2.step(xs, cum_share, where="mid", color=SERIES, linewidth=2, zorder=3)
    ax2.set_title("Cumulative share of patients at or below N surgery-days",
                  fontsize=13, fontweight="600", color=INK, loc="left", pad=10)
    ax2.set_xlabel("distinct days with a surgical procedure", fontsize=10, color=INK_MUTED)
    ax2.set_ylabel("share of patients", fontsize=10, color=INK_MUTED)
    ax2.set_ylim(0, 1.03)
    ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax2.set_xticks(xs)
    ax2.set_xticklabels(labels)
    for level, va in ((0.90, "top"), (0.95, "bottom")):
        ax2.axhline(level, color=GRID, linewidth=1, linestyle="--", zorder=1)
        ax2.text(1.005, level, f"{level:.0%}", transform=ax2.get_yaxis_transform(),
                 va=va, ha="left", fontsize=8, color=INK_MUTED, clip_on=False)

    for a in (ax, ax2):
        a.grid(axis="y", color=GRID, linewidth=1, zorder=0)
        a.set_axisbelow(True)
        for side in ("top", "right"):
            a.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            a.spines[side].set_color(GRID)
        a.tick_params(colors=INK_MUTED, labelsize=9)

    if suppress:
        note = (f"tail aggregated into '{tail_label}' ({n_absorbed} cell(s), "
                f"{tail_patients:,} patients) to meet the {SUPPRESS_THRESHOLD}-patient threshold"
                if tail_label else
                f"no cell below {SUPPRESS_THRESHOLD} patients; nothing aggregated")
    else:
        n_small = int((d["n_patients"] < SUPPRESS_THRESHOLD).sum())
        note = (f"NOT suppressed -- {n_small} cell(s) below {SUPPRESS_THRESHOLD} patients; "
                f"do not share this version")
    fig.text(0.01, 0.005, f"All of Us  |  CDR {CDR}  |  {note}",
             fontsize=8, color=INK_MUTED)

    if save_as:
        out = figure_path(save_as, suppress)
        fig.savefig(out, dpi=150, bbox_inches="tight", facecolor=SURFACE)
        print(f"saved -> {out}"
              + ("   (committable)" if suppress else "   (local only, git-ignored)"))
    return fig

# 2-distribution


### How many patients have each number of surgery-days?

The histogram is built inside BigQuery, returning one row per distinct surgery-day count rather
than one row per patient. Running totals are added so order statistics are recoverable from the
histogram alone, without ever holding per-patient rows.


In [ ]:
distribution_sql = _COHORT_CTES + """
    SELECT
        n_surgery_days,
        COUNT(*) AS n_patients
    FROM surgery_days
    GROUP BY n_surgery_days
    ORDER BY n_surgery_days
"""

surgery_day_dist = cached_gbq(distribution_sql, "surgery_day_distribution")

dist = surgery_day_dist.sort_values("n_surgery_days").reset_index(drop=True)
total_patients = int(dist["n_patients"].sum())
total_surgery_days = int((dist["n_surgery_days"] * dist["n_patients"]).sum())

dist["cum_patients"] = dist["n_patients"].cumsum()
dist["cum_share"] = dist["cum_patients"] / total_patients
dist

### How big is the cohort, and how much of the surgical record is undated?

These denominators matter because a patient can qualify through the index tables yet contribute no
surgery-day, and because rows without a `procedure_datetime` are invisible to a day-based
analysis. A large undated count would undermine any index rule based on dates.


In [ ]:
qc_sql = _COHORT_CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort) AS cohort_patients,
        (SELECT COUNT(*) FROM surgery_days) AS patients_with_dated_surgery,
        (SELECT COUNT(*) FROM surgical_events) AS dated_surgical_rows,
        (SELECT COUNT(*) FROM surgical_concept) AS n_surgical_concepts
"""

qc = cached_gbq(qc_sql, "surgery_day_qc").iloc[0]

missing_sql = _COHORT_CTES + f"""
    SELECT COUNT(*) AS rows_missing_datetime
    FROM `{CDR}`.procedure_occurrence AS po
    WHERE po.person_id IN (SELECT person_id FROM cohort)
      AND po.procedure_concept_id IN (SELECT concept_id FROM surgical_concept)
      AND po.procedure_datetime IS NULL
"""

rows_missing_datetime = int(
    cached_gbq(missing_sql, "surgery_rows_missing_datetime").iloc[0, 0]
)

print(f"cohort patients (index tables)      : {qc['cohort_patients']:>9,}")
print(f"patients with >=1 dated surgery-day : {total_patients:>9,}")
print(f"  ... contributing surgery-days     : {total_surgery_days:>9,}")
print(f"surgical rows lacking a datetime    : {rows_missing_datetime:>9,}")
print(f"distinct surgical concepts in set   : {qc['n_surgical_concepts']:>9,}")

### How concentrated is the distribution?

The median and 90th percentile are read off the cumulative counts, and the split between one and
several surgery-days isolates the population an index rule actually decides anything for. That
group's size sets how much the choice of rule can matter.


In [ ]:
def quantile_from_hist(q):
    """Smallest surgery-day count at which the cumulative share reaches q."""
    idx = (dist["cum_patients"] >= q * total_patients).idxmax()
    return int(dist.loc[idx, "n_surgery_days"])


median_days = quantile_from_hist(0.50)
p90_days = quantile_from_hist(0.90)
max_days = int(dist["n_surgery_days"].max())

print(f"surgery-days per patient  median={median_days}  p90={p90_days}  max={max_days}")

single_day = int(dist.loc[dist["n_surgery_days"] == 1, "n_patients"].sum())
multi_day = total_patients - single_day

print(f"exactly one surgery-day   : {single_day:>9,}  ({single_day / total_patients:6.1%})")
print(f"two or more surgery-days  : {multi_day:>9,}  ({multi_day / total_patients:6.1%})")
print()
print("--> The multi-day share is the population an index-surgery rule actually decides.")

### What does the distribution look like?

The unsuppressed figure is the one to read for the decision and stays git-ignored, while the
suppressed variant aggregates the sparse tail and is the version that may be committed or shared.
Both are written to this notebook's figure folder.


In [ ]:
fig = plot_surgery_days(dist, suppress=False, save_as="surgery_days")
plot_surgery_days(dist, suppress=True, save_as="surgery_days")

if max(dist["n_patients"]) > 50 * min(dist["n_patients"]):
    print("\nTail is orders of magnitude below the mode -- "
          "rerun with logy=True to read it: plot_surgery_days(dist, logy=True)")